In [ ]:
#Importing required modules

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.optim as optim
import torchvision
from torchvision import datasets,transforms
from torch.utils.data import DataLoader


In [ ]:
#Data folder existance
from pathlib import Path
Main_Dir=Path(r"C:\Users\vishn\OneDrive\Desktop\Leaf_Disease_Pediction_DL_Model")
train_dir=Main_Dir/"plantvillage_split"/"train"
test_dir=Main_Dir/"plantvillage_split"/"test"
val_dir=Main_Dir/"plantvillage_split"/"val"
print("Training:", train_dir.exists())
print("Testing:", test_dir.exists())
print("Prediction:", val_dir.exists())

Training: True
Testing: True
Prediction: True


In [ ]:
#Transforming images 


transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))

])
train_data=datasets.ImageFolder(
    root=train_dir,
    transform=transform
)
test_data=datasets.ImageFolder(
    root=test_dir,
    transform=transform
)
validation_data=datasets.ImageFolder(
    root=val_dir,
    transform=transform
)
print("Length of train data= ",len(train_data))
print("Length of test data= ",len(test_data))
print("Length of val data= ",len(validation_data))
print(len(train_data.classes))


Length of train data=  37998
Length of train data=  8162
Length of train data=  8145
38


In [ ]:
#Data loader
traindata=DataLoader(train_data,batch_size=64,shuffle=True)
testdata=DataLoader(test_data,batch_size=64)

In [ ]:
#Class for cnn model with all layers

class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.conv_layers=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.AdaptiveAvgPool2d((4,4))

        )
        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,len(train_data.classes)) 
        )
    def forward(self,x):
        x=self.conv_layers(x)
        x=torch.flatten(x,start_dim=1)
        x=self.fc_layers(x)
        return x
    

In [ ]:
#Creating  object of model, criterion and optimizer
model=CNN()
criterion=nn.CrossEntropyLoss()
optim=optim.Adam(model.parameters())


In [ ]:
#Model training with 5 epochs

train_loss=0.0
val_loss=0.0
epochs=5
for epoch in range(epochs):
    model.train()
    
    train_loss=0.0
    for image,label in traindata:
         optim.zero_grad()
         outputs=model(image)
         loss=criterion(outputs,label)
         loss.backward()
         optim.step()
         train_loss+=loss.item()
    print(f"epoch={epoch+1}/{epochs} & loss={train_loss/len(traindata)}")




epoch=1/5 & loss=0.322438206159908
epoch=2/5 & loss=0.24118305494387945
epoch=3/5 & loss=0.18861874118321872
epoch=4/5 & loss=0.14837269087381635
epoch=5/5 & loss=0.12848328619329918


In [ ]:
#Accuracy Calculation to know the condition of model

model.eval()
correct_labels=0
total_labels=0
with torch.no_grad():
    for image,label in testdata:
        output=model(image)
        _,pred=torch.max(output,1)
        correct_labels += (pred== label).sum().item()
        total_labels += label.size(0)
    print("accuracy=",(correct_labels/total_labels))

correct_labels1=0
total_labels1=0
with torch.no_grad():
    
    for image,label in traindata:
        output=model(image)
        _,pred=torch.max(output,1)
        correct_labels1 += (pred== label).sum().item()
        total_labels1 += label.size(0)
    print("accuracy=",(correct_labels1/total_labels1))

accuracy= 0.9326145552560647
accuracy= 0.9630506868782568


In [ ]:
#Save THe Model
save_dir=Path("model")
save_dir.mkdir(parents=True,exist_ok=True)
torch.save(
    model.state_dict(),
    save_dir/"plant_disease_cnn.pth")
print("Done")

Done


In [54]:
train_data.classes

['Apple___Apple_scab',
 'Apple___Black_rot',
 'Apple___Cedar_apple_rust',
 'Apple___healthy',
 'Blueberry___healthy',
 'Cherry_(including_sour)___Powdery_mildew',
 'Cherry_(including_sour)___healthy',
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot',
 'Corn_(maize)___Common_rust_',
 'Corn_(maize)___Northern_Leaf_Blight',
 'Corn_(maize)___healthy',
 'Grape___Black_rot',
 'Grape___Esca_(Black_Measles)',
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)',
 'Grape___healthy',
 'Orange___Haunglongbing_(Citrus_greening)',
 'Peach___Bacterial_spot',
 'Peach___healthy',
 'Pepper,_bell___Bacterial_spot',
 'Pepper,_bell___healthy',
 'Potato___Early_blight',
 'Potato___Late_blight',
 'Potato___healthy',
 'Raspberry___healthy',
 'Soybean___healthy',
 'Squash___Powdery_mildew',
 'Strawberry___Leaf_scorch',
 'Strawberry___healthy',
 'Tomato___Bacterial_spot',
 'Tomato___Early_blight',
 'Tomato___Late_blight',
 'Tomato___Leaf_Mold',
 'Tomato___Septoria_leaf_spot',
 'Tomato___Spider_mites Two-spotted_